# 03 — Agent Definition & Evaluation
**Meridian Governance Group — AI Policy Research Agent**

Owner role: **AI Engineer (AIE)**

This notebook:
1. Defines the ReAct agent (LLM + 3 tools + out-of-scope handling).
2. Starts **Arize Phoenix** tracing and instruments Anthropic calls.
3. Runs **5 evaluation traces**, including one **head-to-head trace running two LLMs on the same query**.
4. Demonstrates **2 graceful rejections** of out-of-scope queries.
5. Scores every response with an **LLM judge** and writes commentary.
6. Computes an **ROI comparison** of the two models.

> **AI usage disclosure:** Scaffolding developed with Anthropic Claude (Claude Code); all evaluation design reviewed by the author. The presentation video contains **no AI usage** and is the author's own work.

In [ ]:
%pip install anthropic openai sentence-transformers python-dotenv mlflow matplotlib seaborn pandas typing-extensions databricks-agents databricks-openai databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
import sys, pathlib, os, json
sys.path.append(str(pathlib.Path.cwd()))

from dotenv import load_dotenv
load_dotenv()  # loads ANTHROPIC_API_KEY (and DATABRICKS_* vars) from .env if present

# Try UCFunctionToolkit for Databricks Unity Catalog tool integration.
# Falls back gracefully on package incompatibility (VectorSearchIndex conflict).
try:
    from databricks_openai import UCFunctionToolkit
    USE_DATABRICKS_OPENAI = True
except ImportError as e:
    if "VectorSearchIndex" in str(e):
        print(f"Warning: databricks_openai import failed — using fallback.\nError: {e}")
        USE_DATABRICKS_OPENAI = False
    else:
        raise

import mlflow
from sentence_transformers import SentenceTransformer

from src import config
from src.vector_store import SimpleVectorStore
from src.tools import PolicyToolbox
from src import evaluation as ev

# Anthropic is optional — only needed if ANTHROPIC_API_KEY is set.
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

# Build the judge client once; used in every ev.judge_response() call below.
# Returns an Anthropic client if the key is set, otherwise a Databricks client.
judge_client = config.get_judge_client()
print(f"Judge backend: {config.JUDGE_BACKEND}  model: {config.JUDGE_MODEL}")


In [ ]:
import pathlib, importlib, sys

# Clear stale bytecode so re-uploaded src files are always picked up fresh.
# Run this cell any time you upload new versions of src/*.py without restarting.
for pyc in pathlib.Path("src/__pycache__").glob("*.pyc"):
    pyc.unlink(missing_ok=True)
for key in list(sys.modules):
    if key.startswith("src"):
        del sys.modules[key]

print("Module cache cleared.")

## 1. Start Arize Phoenix tracing
Phoenix is an open-source, locally-hosted trace provider (an explicit alternative to Databricks MLflow). `register()` launches the local app and the Anthropic instrumentor records every model call as a span. Open the printed URL to inspect the 5 traces live.

In [ ]:
# MLflow is pre-installed on Databricks; no extra pip install needed.
# Autolog instruments every Anthropic AND OpenAI-compatible call automatically.
mlflow.set_experiment("/meridian-policy-agent")
mlflow.anthropic.autolog()
mlflow.openai.autolog()   # covers DatabricksAgent and OpenAICompatAgent
print("MLflow autologging active. View traces in the Experiments UI.")


## 2. Load the knowledge base and build the agent
We reuse the vector store from notebook 01 and the same local embedder.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd()))

from databricks.vector_search.client import VectorSearchClient
from src.tools import DatabricksVSToolbox

vs_index = VectorSearchClient().get_index(
    "ai_governance_endpoint", "main.default.ai_governance_index"
)
toolbox = DatabricksVSToolbox(vs_index)
print("Toolbox ready — using Databricks Vector Search.")

# Build every LLM defined in the registry so they are ready for the traces.
agents = {key: config.create_agent(key, toolbox) for key in config.LLM_REGISTRY}

db_primary_agent   = agents["db_primary"]
db_secondary_agent = agents["db_secondary"]
opensource_agent   = agents["opensource"]
primary_agent      = agents.get("sonnet", db_primary_agent)

for key, agent in agents.items():
    label = config.LLM_REGISTRY[key]["label"]
    model_id = getattr(agent, "model", "(auto)")
    print(f"  {key:14s} → {label} ({model_id})")


## 3. The five evaluation traces
Trace 1 is the **head-to-head** (both models, same query). Traces 2-3 are further in-scope questions on the primary model. Traces 4-5 are **out-of-scope** queries the agent must decline. Each `agent.run(...)` is captured by Phoenix as a trace.

### Trace 1 — head-to-head: three LLMs, same query
Satisfies the rubric's requirement to use at least two different LLMs within the same trace — here we run **three**: the primary Claude model (`claude-sonnet-4-6`), the secondary Claude model (`claude-haiku-4-5`), and an **open-source Qwen3 model self-hosted on llama.cpp** (reached via its OpenAI-compatible API). All three answer the identical question inside one parent span, so the judge and ROI step can compare them directly.

In [ ]:
hh = ev.IN_SCOPE_QUERIES[0]  # the head-to-head query
print("QUERY:", hh["query"], "\n")

hh_keys   = config.EVAL_HEAD_TO_HEAD_KEYS
hh_labels = [config.LLM_REGISTRY[k]["label"] for k in hh_keys]

hh_results = {}
hh_errors  = {}
with mlflow.start_run(run_name="head_to_head_multi_llm") as run:
    mlflow.set_tags({
        "eval.query_id": hh["id"],
        "eval.models": ", ".join(hh_labels),
    })
    for key in hh_keys:
        label = config.LLM_REGISTRY[key]["label"]
        try:
            hh_results[key] = agents[key].run(hh["query"])
            mlflow.log_metric(f"latency_s_{key}", hh_results[key].latency_s)
            print(f"OK   {label}")
        except Exception as e:
            hh_errors[key] = e
            print(f"FAIL {label}: {type(e).__name__}: {e}")

print(f"\nMLflow run: {run.info.run_id}\n")
for key, res in hh_results.items():
    print(f"--- {config.LLM_REGISTRY[key]['label']} ---")
    print(res.answer)
    print(f"[tools: {[t['name'] for t in res.tool_calls]}, "
          f"tokens in/out: {res.input_tokens}/{res.output_tokens}, "
          f"latency: {res.latency_s:.1f}s]\n")

if hh_errors:
    print("Failed models (skipped in evaluation):")
    for key, err in hh_errors.items():
        print(f"  {config.LLM_REGISTRY[key]['label']}: {type(err).__name__}: {err}")

# Only keep keys that succeeded; downstream cells use hh_keys.
hh_keys = list(hh_results.keys())

db_primary_result   = hh_results.get("db_primary")
db_secondary_result = hh_results.get("db_secondary")
opensource_result   = hh_results.get("opensource")
primary_result      = hh_results.get("sonnet", db_primary_result)
secondary_result    = hh_results.get("haiku", db_secondary_result)


### Traces 2-3 — additional in-scope questions (primary model)

In [ ]:
in_scope_results = {hh['id']: primary_result}
for q in ev.IN_SCOPE_QUERIES[1:]:
    print("QUERY:", q["query"])
    res = primary_agent.run(q["query"])
    in_scope_results[q["id"]] = res
    print(res.answer)
    print(f"[tools: {[t['name'] for t in res.tool_calls]}]\n" + "-" * 80)

### Traces 4-5 — graceful rejection of out-of-scope queries
The agent must recognize these are unrelated to AI policy and decline politely, redirecting to what it can help with — **without** answering the off-topic question. This is the required error/out-of-scope handling and the 2 rejection examples.

In [ ]:
oos_results = {}
for q in ev.OUT_OF_SCOPE_QUERIES:
    print("QUERY:", q["query"])
    res = primary_agent.run(q["query"])
    oos_results[q["id"]] = res
    print("RESPONSE:", res.answer)
    print(f"[tools used: {[t['name'] for t in res.tool_calls]} "
          f"(expected: none)]\n" + "-" * 80)

## 4. LLM-as-judge scoring
We score every response on accuracy, relevance, completeness, and clarity (1-5). The judge is told that for out-of-scope queries a clean refusal is the *correct* behavior and should score high.

In [ ]:
rows = []
all_results = {
    **{qid: r for qid, r in in_scope_results.items()},
    **{qid: r for qid, r in oos_results.items()},
}
query_by_id = {q['id']: q for q in ev.ALL_QUERIES}

judged = {}
for qid, res in all_results.items():
    meta = query_by_id[qid]
    score = ev.judge_response(judge_client, res.query, res.answer, meta['category'])
    judged[qid] = score
    rows.append({
        "id": qid, "category": meta['category'], "model": res.model,
        "accuracy": score.accuracy, "relevance": score.relevance,
        "completeness": score.completeness, "clarity": score.clarity,
        "overall": round(score.overall, 2),
        "cost_usd": round(ev.cost_of_run(res), 6),
        "latency_s": round(res.latency_s, 1),
    })

# Judge head-to-head models — skip any that failed (result is None).
db_primary_score   = ev.judge_response(judge_client, db_primary_result.query,   db_primary_result.answer,   'in_scope') if db_primary_result   else None
db_secondary_score = ev.judge_response(judge_client, db_secondary_result.query, db_secondary_result.answer, 'in_scope') if db_secondary_result else None
opensource_score   = ev.judge_response(judge_client, opensource_result.query,   opensource_result.answer,   'in_scope') if opensource_result   else None

primary_score   = judged.get(hh['id'], db_primary_score)
secondary_score = db_secondary_score

import pandas as pd
scores_df = pd.DataFrame(rows)
scores_df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
dims = ["accuracy", "relevance", "completeness", "clarity"]
fig, ax = plt.subplots(figsize=(10, 5))
scores_df.set_index("id")[dims].plot(kind="bar", ax=ax)
ax.set_title("LLM-judge scores by query and dimension")
ax.set_ylim(0, 5.4); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 5. Head-to-head judge comparison + written commentary
Compare the two models on the identical query.

In [ ]:
for key, res, sc in [
    ("db_primary",   db_primary_result,   db_primary_score),
    ("db_secondary", db_secondary_result, db_secondary_score),
    ("opensource",   opensource_result,   opensource_score),
] + ([("sonnet", hh_results.get("sonnet"), judged.get(hh["id"]))] if "sonnet" in hh_results else []):
    if res is None or sc is None:
        print(f"{config.LLM_REGISTRY[key]['label']}  — skipped (failed to run)")
        continue
    print(f"{config.LLM_REGISTRY[key]['label']}  ({res.model})")
    print(f"  overall={sc.overall:.2f}  rationale: {sc.rationale}")


**Commentary (author):**

_Fill this in after running, based on the judge scores above._ Typical pattern: the larger primary model (Sonnet) scores higher on completeness and accuracy because it cites more specific articles/functions and integrates evidence from multiple tool calls, while the smaller secondary model (Haiku) is faster and far cheaper but tends to be terser and cite fewer specifics. Note any concrete differences you observe in the two answers and which dimensions drove the gap.

_Also compare the open-source Qwen3 model:_ note whether it used the tools natively or fell back to retrieval, and how its accuracy/citations compare to the Claude models — this is the crux of the build-vs-buy argument for Meridian.

## 6. ROI calculation — which model should Meridian deploy?
ROI here is **judged quality per dollar**. We combine the judge's overall quality with the measured per-run cost (computed from token usage at Anthropic's published per-million-token pricing).

In [ ]:
roi = ev.roi_comparison(primary_result, secondary_result, primary_score, secondary_score)
print(json.dumps(roi, indent=2, default=str))

In [ ]:
p, s = roi['primary'], roi['secondary']
print(f"Cost: {p['model']} = ${p['cost_usd']:.6f}  vs  {s['model']} = ${s['cost_usd']:.6f}")
print(f"  -> primary costs {roi['cost_ratio_primary_over_secondary']:.1f}x the secondary")
print(f"Quality (judge overall): {p['overall_quality']:.2f} vs {s['overall_quality']:.2f} "
      f"(delta {roi['quality_delta_primary_minus_secondary']:+.2f})")
print(f"Quality per dollar: {p['quality_per_dollar']:.0f} vs {s['quality_per_dollar']:.0f}")

QUERIES_PER_MONTH = 5000
print(f"\nProjected monthly cost @ {QUERIES_PER_MONTH:,} queries/month:")
print(f"  {p['model']}:  ${p['cost_usd'] * QUERIES_PER_MONTH:,.2f}")
print(f"  {s['model']}:  ${s['cost_usd'] * QUERIES_PER_MONTH:,.2f}")
print(f"  monthly savings with secondary: "
      f"${(p['cost_usd'] - s['cost_usd']) * QUERIES_PER_MONTH:,.2f}")

# Multi-way ROI table — only include models that successfully ran and were judged.
import pandas as pd
all_hh_results, all_hh_scores = {}, {}
for k, res, sc in [
    ("db_primary",   db_primary_result,   db_primary_score),
    ("db_secondary", db_secondary_result, db_secondary_score),
    ("opensource",   opensource_result,   opensource_score),
] + ([("sonnet", hh_results.get("sonnet"), judged.get(hh["id"]))] if "sonnet" in hh_results else []):
    if res is not None and sc is not None:
        label = config.LLM_REGISTRY[k]["label"]
        all_hh_results[label] = res
        all_hh_scores[label]  = sc

roi_df = pd.DataFrame(ev.roi_table(all_hh_results, all_hh_scores))
print("\nFull model comparison:")
print(roi_df.to_string(index=False))
print("\nNote: Databricks and self-hosted models show cost $0 — marginal per-token "
      "cost is ~$0; real cost is fixed cluster/GPU infra.")


**Deployment recommendation (author):**

_Write your recommendation here after reviewing the numbers._ Frame it for Meridian: if the quality gap on in-scope policy questions is small relative to the multiple-times cost difference, recommend the cheaper model for routine queries and reserve the primary model for high-stakes client deliverables (a tiered/human-in-the-loop approach). If accuracy on regulatory specifics materially differs, justify paying for the primary model given the compliance risk of a wrong answer.

With the open-source model in the mix, frame an explicit **build-vs-buy** decision: a self-hosted Qwen3 has ~zero marginal cost but carries fixed infra + ops + data-governance burden, and may trail on regulatory accuracy. Recommend where each model fits (e.g. open-source for high-volume internal drafting where data must stay on-prem; Claude for client-facing compliance answers where accuracy is paramount).

## 7. Persist evaluation artifacts
Save all results + judge scores to `traces/` for the report and video.

In [ ]:
records = []
for qid, res in all_results.items():
    records.append(ev.result_to_record(res, judged.get(qid)))
records.append(ev.result_to_record(secondary_result, secondary_score))
records.append(ev.result_to_record(opensource_result, opensource_score))

out = config.TRACES_DIR / "evaluation_results.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump({"results": records, "roi": roi}, f, indent=2, default=str)
scores_df.to_csv(config.TRACES_DIR / "judge_scores.csv", index=False)
print("saved:", out)
print("saved:", config.TRACES_DIR / "judge_scores.csv")

### Evaluation complete
You now have: 5 Phoenix traces (incl. a 2-LLM head-to-head), 2 graceful rejections, LLM-judge scores with commentary, and an ROI-based deployment recommendation. Screenshots of the Phoenix UI traces belong in the report/video.